# Spark Interoperability with the Fleet Iceberg V3 Tables

This notebook queries the **Smart Fleet** Snowflake-managed Iceberg V3 tables from Apache Spark
through the Snowflake **Horizon Iceberg REST catalog**, with **access controls enforced by
Snowflake** (the same masking policies apply cross-engine).

It demonstrates:
- Reading the fleet tables created by the `001`–`010` SQL pipeline via the REST catalog
- `variant_get` over the `TELEMETRY_DATA` VARIANT column
- **Enforced governance**: the same `VEHICLE_REGISTRY` query returns full PII as the engineer
  role and *masked* PII as `FLEET_ANALYST` — enforced by Horizon, not by Spark

## Prerequisites
1. Completed the Snowflake setup (`task demo-up`) so the fleet tables and masking policies exist
2. Apache **Spark 4.0+** (required for VARIANT support) and **Java 17+**
3. A named `snow` CLI connection (set via `SPARK_CLI_CONNECTION_NAME`) configured with
   **key-pair auth** — the notebook reads `account`, `user`, and `role` from this connection
   in `~/.snowflake/connections.toml` or `config.toml`, and mints a short-lived JWT with
   `snow connection generate-jwt` for the Horizon REST catalog (no PAT required)
4. The connection user's password exported for the `spark-snowflake` connector, following the
   snow CLI convention: `export SNOWFLAKE_CONNECTIONS_<CONNECTION_NAME>_PASSWORD="..."`
5. Config is read from environment variables exported from `.env/iceberg.env`

## Configuration

Connection metadata (`account`, `user`, `role`) is resolved from the named `snow` CLI
connection in `SPARK_CLI_CONNECTION_NAME`; the Horizon REST catalog URI is derived from the
account. The Horizon REST catalog authenticates with a key-pair JWT minted from that
connection (no PAT). The `spark-snowflake` connector authenticates with the connection's user
and the password from `SNOWFLAKE_CONNECTIONS_<CONNECTION_NAME>_PASSWORD`.

In [ ]:
import os
import subprocess
import time
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

try:
    import tomllib  # Python 3.11+ stdlib
except ModuleNotFoundError:
    import tomli as tomllib  # type: ignore[no-redef]


@contextmanager
def step(label):
    """Print clear START / DONE / FAILED markers with elapsed time so you can
    tell from the Jupyter output when a cell started and whether it finished.
    """
    start = time.perf_counter()
    print(f"\u25b6 START  {label}  [{datetime.now():%H:%M:%S}]", flush=True)
    try:
        yield
    except Exception as exc:
        elapsed = time.perf_counter() - start
        print(f"\u2716 FAILED {label}  after {elapsed:,.1f}s  ({type(exc).__name__}: {exc})", flush=True)
        raise
    else:
        elapsed = time.perf_counter() - start
        print(f"\u2714 DONE   {label}  in {elapsed:,.1f}s", flush=True)


def _load_named_connection(name: str) -> dict:
    """Return the params for the named ``snow`` CLI connection.

    Search order:
      1. ``~/.snowflake/connections.toml``  (each connection is a top-level table)
      2. ``~/.snowflake/config.toml``       (each connection under ``[connections.<name>]``)

    Raises ``RuntimeError`` if neither file contains the named connection.
    """
    home = Path.home() / ".snowflake"
    candidates = [home / "connections.toml", home / "config.toml"]
    for path in candidates:
        if not path.exists():
            continue
        with path.open("rb") as fh:
            data = tomllib.load(fh)
        # Layout 1 (connections.toml): [<name>]  <-- top-level table
        if name in data and isinstance(data[name], dict):
            return dict(data[name])
        # Layout 2 (config.toml): [connections.<name>]
        connections = data.get("connections", {})
        if name in connections:
            return dict(connections[name])
    raise RuntimeError(
        f"Connection '{name}' not found in {candidates[0]} or {candidates[1]}"
    )


def generate_jwt(connection: str) -> str:
    """Mint a key-pair JWT for the named snow CLI connection (valid ~60 min).

    Shells out to `snow connection generate-jwt`, which signs a JWT with the
    private key already configured in the connection. No PAT required.
    """
    result = subprocess.run(
        ["snow", "connection", "generate-jwt", "--connection", connection, "--silent"],
        capture_output=True,
        text=True,
        check=True,
    )
    token = result.stdout.strip()
    if not token:
        raise RuntimeError("snow connection generate-jwt returned empty output")
    return token


# --- Named snow CLI connection (drives account / user / role) ---
# account, user, and role are read from this connection in
# ~/.snowflake/connections.toml or ~/.snowflake/config.toml.
cli_connection = os.environ["SPARK_CLI_CONNECTION_NAME"]
_conn = _load_named_connection(cli_connection)

account = _conn["account"]
user = _conn["user"]
# Connection's default role; used as the connector role unless build_spark()
# is called with an explicit role (the masking demo passes engineer/analyst).
connection_role = _conn.get("role")

# Password for the spark-snowflake connector. The Horizon REST catalog still
# authenticates with the key-pair JWT (generate_jwt); the connector needs a
# user/password. Follows the snow CLI env-var override convention:
#   export SNOWFLAKE_CONNECTIONS_<CONNECTION_NAME>_PASSWORD="..."
_pw_var = f"SNOWFLAKE_CONNECTIONS_{cli_connection.upper()}_PASSWORD"
try:
    snowflake_password = os.environ[_pw_var]
except KeyError:
    raise RuntimeError(
        f"Password env var {_pw_var} is not set. Export it, e.g.:\n"
        f'    export {_pw_var}="<your_password>"'
    )

# --- Horizon REST catalog (derived from the connection account) ---
# Format: https://<account_identifier>.snowflakecomputing.com/polaris/api/catalog
sf_url = f"{account}.snowflakecomputing.com"
horizon_catalog_uri = f"https://{sf_url}/polaris/api/catalog"

# The Snowflake database (Iceberg catalog/warehouse name).
database_name = os.environ["DEMO_DATABASE_NAME"]

# Bronze schema (Iceberg namespace) holding the fleet tables.
bronze_schema = os.environ.get("DEMO_SCHEMA_NAME_BRONZE", "RAW")

# Warehouse for the spark-snowflake connector.
warehouse = os.environ.get("DEMO_WAREHOUSE_NAME", "FLEET_ANALYTICS_WH")

# Roles for the enforced-masking demo.
analyst_role = os.environ.get("DEMO_ANALYST_ROLE_NAME", "FLEET_ANALYST")
engineer_role = os.environ.get("DEMO_ENGINEER_ROLE_NAME", "V3_DEMO_ICEBERG_ENGINEER_ROLE")

# Cloud provider selects the Iceberg cloud SDK bundle + FileIO. Even with
# Snowflake-managed storage, Spark reads data files through cloud SDK bundles,
# so match this to your Snowflake account's cloud (aws | gcp | azure).
cloud_provider = os.environ.get("SPARK_CLOUD_PROVIDER", "aws").lower()
aws_region = os.environ.get("AWS_REGION", "us-east-1")

# Iceberg runtime + cloud bundle version (Spark 4.0 / Scala 2.13).
iceberg_version = os.environ.get("SPARK_ICEBERG_VERSION", "1.10.1")
scala_version = "2.13"

# spark-snowflake connector + JDBC driver versions.
snowflake_jdbc_version = "3.24.0"
snowflake_spark_connector_version = "3.1.6"

with step("Print Initial Output"):
    print(f"    CLI connection:     {cli_connection}")
    print(f"    Account:            {account}")
    print(f"    User:               {user}")
    print(f"    Connection role:    {connection_role}")
    print(f"    Catalog URI:        {horizon_catalog_uri}")
    print(f"    Snowflake Database: {database_name}")
    print(f"    Bronze schema:      {bronze_schema}")
    print(f"    Warehouse:          {warehouse}")
    print(f"    Analyst role:       {analyst_role}")
    print(f"    Engineer role:      {engineer_role}")
    print(f"    Cloud provider:     {cloud_provider}")
    print(f"    AWS region:         {aws_region}")
    print(f"    Iceberg version:    {iceberg_version}")

In [2]:
# Map cloud provider -> (Iceberg cloud bundle, FileIO implementation).
_CLOUD = {
    "aws": ("org.apache.iceberg:iceberg-aws-bundle", "org.apache.iceberg.aws.s3.S3FileIO"),
    "gcp": ("org.apache.iceberg:iceberg-gcp-bundle", "org.apache.iceberg.gcp.gcs.GCSFileIO"),
    "azure": ("org.apache.iceberg:iceberg-azure-bundle", "org.apache.iceberg.azure.adlsv2.ADLSFileIO"),
}
if cloud_provider not in _CLOUD:
    raise ValueError(f"SPARK_CLOUD_PROVIDER must be one of {list(_CLOUD)}; got {cloud_provider!r}")

bundle_pkg, file_io = _CLOUD[cloud_provider]
bundle = f"{bundle_pkg}:{iceberg_version}"

with step("Print Bundle + FileIO Output"):
    print(f"    Using bundle: {bundle}")
    print(f"    Using FileIO: {file_io}")

▶ START  Print Bundle + FileIO Output  [13:09:57]
    Using bundle: org.apache.iceberg:iceberg-aws-bundle:1.10.1
    Using FileIO: org.apache.iceberg.aws.s3.S3FileIO
✔ DONE   Print Bundle + FileIO Output  in 0.0s


## Create a Spark session scoped to a Snowflake role

`build_spark(role)` creates a Spark session authenticated to the Horizon REST catalog as the
given role. Because Snowflake enforces access controls (including masking policies) based on
this role, switching roles changes what the **same** query returns — the enforcement happens
in Snowflake, not in Spark.

In [ ]:
from pyspark.sql import SparkSession


def build_spark(role=None):
    """Create a Spark session authed to the Horizon REST catalog as ``role``.

    The Horizon REST catalog authenticates with a key-pair JWT minted from the
    named CLI connection; the spark-snowflake connector authenticates with the
    connection's user + password. Snowflake enforces access controls based on
    ``role``, so switching roles changes what the same query returns.

    Notes:
    - driver.host / bindAddress pin Spark to localhost (avoids VPN issues)
    - Spark 4.0 + Iceberg for native VARIANT support
    """
    role = role or connection_role
    if role is None:
        raise ValueError(
            "No role available: pass a role to build_spark() or set 'role' "
            f"on the '{cli_connection}' connection."
        )

    # Stop any active session so a role switch actually re-authenticates.
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()

    # Key-pair JWT is the Horizon REST catalog credential (no PAT).
    token = generate_jwt(cli_connection)

    spark = SparkSession.builder \
        .appName("Fleet Analytics - Iceberg V3 Interop") \
        .master("local[*]") \
        .config("spark.driver.host", "127.0.0.1") \
        .config("spark.driver.bindAddress", "127.0.0.1") \
        .config("spark.jars.packages",
                f"org.apache.iceberg:iceberg-spark-runtime-4.0_{scala_version}:{iceberg_version},"
                f"{bundle},"
                f"net.snowflake:snowflake-jdbc:{snowflake_jdbc_version},"
                f"net.snowflake:spark-snowflake_{scala_version}:{snowflake_spark_connector_version}") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.catalog.horizon", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.horizon.type", "rest") \
        .config("spark.sql.catalog.horizon.uri", horizon_catalog_uri) \
        .config("spark.sql.catalog.horizon.credential", token) \
        .config("spark.sql.catalog.horizon.warehouse", database_name) \
        .config("spark.sql.catalog.horizon.scope", f"session:role:{role}") \
        .config("spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation", "vended-credentials") \
        .config("spark.snowflake.sfURL", sf_url) \
        .config("spark.snowflake.sfUser", user) \
        .config("spark.snowflake.sfPassword", snowflake_password) \
        .config("spark.snowflake.sfDatabase", database_name) \
        .config("spark.snowflake.sfSchema", bronze_schema) \
        .config("spark.snowflake.sfRole", role) \
        .config("spark.snowflake.sfWarehouse", warehouse) \
        .config("spark.sql.iceberg.vectorization.enabled", "false") \
        .getOrCreate()

    print(f"Spark session created as role '{role}' (Spark {spark.version})")
    return spark


print("build_spark(role) is ready. Call build_spark(engineer_role) to start.")

## 1. Connect as the engineer role (full access) and list the fleet tables

In [ ]:
spark = build_spark(engineer_role)

with step("List fleet namespaces and tables"):
    spark.sql("SHOW NAMESPACES").show()
    spark.sql(f"SHOW TABLES IN {bronze_schema}").show(truncate=False)

## 2. Query the VARIANT telemetry column with `variant_get`

`VEHICLE_TELEMETRY_STREAM.TELEMETRY_DATA` is a VARIANT (Iceberg V3). Spark 4.0 reads it
natively and `variant_get` extracts nested fields by JSON path.

In [ ]:
with step("Query telemetry VARIANT with variant_get"):
    spark.sql(f"""
SELECT
    VEHICLE_ID,
    EVENT_TIMESTAMP,
    variant_get(TELEMETRY_DATA, '$.speed_mph', 'double')                 AS speed_mph,
    variant_get(TELEMETRY_DATA, '$.engine.rpm', 'int')                   AS engine_rpm,
    variant_get(TELEMETRY_DATA, '$.engine.fuel_level_pct', 'double')     AS fuel_level_pct,
    variant_get(TELEMETRY_DATA, '$.diagnostics.check_engine', 'boolean') AS check_engine,
    variant_get(TELEMETRY_DATA, '$.metadata.region', 'string')          AS region
FROM {bronze_schema}.VEHICLE_TELEMETRY_STREAM
ORDER BY EVENT_TIMESTAMP DESC
LIMIT 20
""").show(truncate=False)

## 3. Enforced governance — full PII as the engineer role

`VEHICLE_REGISTRY` carries driver PII (`DRIVER_NAME`, `DRIVER_EMAIL`, `DRIVER_PHONE`) protected
by Snowflake masking policies. The engineer role is exempt, so it sees the raw values — even
from Spark.

In [ ]:
with step("Read VEHICLE_REGISTRY PII as engineer role (full)"):
    spark.sql(f"""
SELECT VEHICLE_ID, DRIVER_NAME, DRIVER_EMAIL, DRIVER_PHONE
FROM {bronze_schema}.VEHICLE_REGISTRY
ORDER BY VEHICLE_ID
LIMIT 10
""").show(truncate=False)

## 4. Same query as `FLEET_ANALYST` — PII is masked

Reconnect as the analyst role and run the **identical** query. The masking policies are applied
by Horizon, so the PII columns come back masked. Nothing changed in Spark — only the role.

In [ ]:
spark = build_spark(analyst_role)

with step("Read VEHICLE_REGISTRY PII as FLEET_ANALYST (masked)"):
    spark.sql(f"""
SELECT VEHICLE_ID, DRIVER_NAME, DRIVER_EMAIL, DRIVER_PHONE
FROM {bronze_schema}.VEHICLE_REGISTRY
ORDER BY VEHICLE_ID
LIMIT 10
""").show(truncate=False)

## Summary

Apache Spark read the **same** Snowflake-managed Iceberg V3 fleet tables through the Horizon
REST catalog with vended credentials, queried VARIANT data with `variant_get`, and saw
Snowflake's masking policies **enforced consistently across engines** — full PII for the
engineer role, masked PII for `FLEET_ANALYST`. Governance lives with the data, not the engine.